In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import os
import json
import warnings
warnings.filterwarnings("ignore")

In [2]:
with open("../parameters/flowParams.json") as f:
    flowParams = json.load(f)

Mach_inf = flowParams["Mach_inf"]
rho_inf = flowParams["rho_inf"]
p_inf = flowParams["p_inf"]

with open("../parameters/bodyParams.json") as f:
    bodyParams = json.load(f)

transitionPoint = bodyParams["transitionPoint"]
bodyLength = bodyParams["bodyLength"]
bodyType = bodyParams["bodyType"]

with open("../parameters/NumericalParams.json") as f:
    NumericalParams = json.load(f)

(
    NumericalParams["N"],
    NumericalParams["M"]
)

(1600, 4800)

In [3]:
# φ и p(φ), ρ(φ), V_R(φ), V_θ(φ)
initial_data_root = "../initial_cone_flow/output_results/"
data_p   = np.loadtxt(initial_data_root + 'p_init.txt')            # shape (L,2): [φ, p]
data_rho = np.loadtxt(initial_data_root + 'rho_init.txt')          # shape (L,2): [φ, ρ]
data_vr_vtheta = np.loadtxt(initial_data_root + 'VR_Vtheta_init.txt')

phi_cone, p_cone = data_p[::-1,0], data_p[::-1,1]
_, rho_cone = data_rho[::-1,0], data_rho[::-1,1]
_, VR_cone, Vtheta_cone = data_vr_vtheta[::-1,0], data_vr_vtheta[::-1,1], data_vr_vtheta[::-1,2]

In [4]:
def r_b(z):
    if bodyType == "Cylindrical":
        return np.tan(np.pi / 12)
    elif bodyType == "Cone":
        return np.tan(np.pi / 12) * z
    elif bodyType == "Parabolic":
        return np.tan(np.pi / 12) * np.sqrt(2*z - 1)
    elif bodyType == "DoubleCone":
        double_cone_k = 0.01
        return np.tan(np.pi / 12) + double_cone_k * (z - 1)

In [5]:
def delta_by_C_norm(a, b):
    return np.max(np.abs(a - b))

In [6]:
def delta_by_L1_norm(a, b, r_s, z):
    N, M = a.shape
    dr = (r_s - r_b(z)) / N
    dtheta = np.pi / M
    r = np.zeros_like(a)
    for i in range(N):
        for j in range(M):
            r[i, j] = r_b(z) + i * dr[j]
    return np.sum(np.abs(a - b) * r * dr[np.newaxis, :] * dtheta)

### Сходимость по сетке для автомодельной задачи обтекания конуса (начальное условие основной задачи)

In [7]:
# φ и p(φ), ρ(φ), V_R(φ), V_θ(φ)
initial_data_root = "../initial_cone_flow/output_results/"
data_p   = np.loadtxt(initial_data_root + 'p_init.txt')            # shape (L,2): [φ, p]
data_rho = np.loadtxt(initial_data_root + 'rho_init.txt')          # shape (L,2): [φ, ρ]
data_vr_vtheta = np.loadtxt(initial_data_root + 'VR_Vtheta_init.txt')

phi_cone, p_cone = data_p[::-1,0], data_p[::-1,1]
_, rho_cone = data_rho[::-1,0], data_rho[::-1,1]
_, VR_cone, Vtheta_cone = data_vr_vtheta[::-1,0], data_vr_vtheta[::-1,1], data_vr_vtheta[::-1,2]

### Сходимость по сетке для основной задачи

In [8]:
data_dict = {}

In [11]:
for N_nodes_count in [100, 200, 400, 800, 1600]:
    data_root = f"analytical_cone_fixed/N{N_nodes_count}/"
    
    zs = np.loadtxt(data_root + 'z_out.txt')
    rs = np.loadtxt(data_root + 'r_s_out.txt')
    Fy = np.loadtxt(data_root + 'Fy_out.txt')
    Mz = np.loadtxt(data_root + 'Mz_out.txt')
    K, M = rs.shape
    rho = np.loadtxt(data_root + 'rho_out.txt')
    assert rho.shape[0] % K == 0
    N = rho.shape[0] // K
    rho = rho.reshape(K, N, M)
    p = np.loadtxt(data_root + 'p_out.txt').reshape(K, N, M)
    u = np.loadtxt(data_root + 'u_out.txt').reshape(K, N, M)
    v = np.loadtxt(data_root + 'v_out.txt').reshape(K, N, M)
    w = np.loadtxt(data_root + 'w_out.txt').reshape(K, N, M)

    data_dict[N_nodes_count] = (zs, rs, rho, p, u, v, w, Fy, Mz)
    print('Количество узлов:')
    print(f'z: {K}, r: {N}, theta: {M}')
    print(f'успешно загружено для {N = }')

Количество узлов:
z: 101, r: 100, theta: 300
успешно загружено для N = 100
Количество узлов:
z: 101, r: 200, theta: 600
успешно загружено для N = 200
Количество узлов:
z: 101, r: 400, theta: 1200
успешно загружено для N = 400
Количество узлов:
z: 101, r: 800, theta: 2400
успешно загружено для N = 800
Количество узлов:
z: 101, r: 1600, theta: 4800
успешно загружено для N = 1600


In [9]:
def map_to_nearest(a, x, y):
    """
    Для каждого элемента a выбирает y[idx], где idx – индекс ближайшего значения в x.
    
    Параметры:
    a : np.ndarray, форма (N, M)
    x : np.ndarray, форма (K,), монотонный (возрастает или убывает)
    y : np.ndarray, форма (K,)
    
    Возвращает:
    np.ndarray формы (N, M)
    """
    # Обработка убывающего x (инвертируем, чтобы был возрастающим)
    if x[0] > x[-1]:
        x = x[::-1]
        y = y[::-1]
    
    a_flat = a.ravel()
    # Поиск позиций вставки (левая граница)
    pos = np.searchsorted(x, a_flat)  # форма (N*M,)
    
    # Инициализация массива индексов ближайших элементов
    closest_idx = np.empty_like(pos, dtype=int)
    
    # Граничные случаи
    left_bound = (pos == 0)
    right_bound = (pos == len(x))
    mid = ~(left_bound | right_bound)
    
    closest_idx[left_bound] = 0
    closest_idx[right_bound] = len(x) - 1
    
    # Внутренние случаи: сравниваем расстояния
    if np.any(mid):
        pos_mid = pos[mid]
        a_mid = a_flat[mid]
        # Расстояния до левого и правого соседа
        dist_left = a_mid - x[pos_mid - 1]
        dist_right = x[pos_mid] - a_mid
        # Выбираем левого, если расстояние до него не больше
        choose_left = dist_left <= dist_right
        closest_idx_mid = np.where(choose_left, pos_mid - 1, pos_mid)
        closest_idx[mid] = closest_idx_mid
    
    # Берём значения из y
    result_flat = y[closest_idx]
    return result_flat.reshape(a.shape)

In [12]:
for N in data_dict.keys():
    t = np.linspace(0, 1, N)[:, np.newaxis]

    # Результирующий массив формы (200, 600)
    result = r_b(data_dict[N][0][-1]) + (data_dict[N][1][-1,:] - r_b(data_dict[N][0][-1])) * t

    rho_analytical = map_to_nearest(np.arctan(result / zs[-1]), phi_cone, rho_cone)

    print(f'{N = }')
    print(f'Delta_C = {delta_by_C_norm(data_dict[N][2][-1,:,:], rho_analytical)}')
    print(f'Delta_L = {delta_by_L1_norm(data_dict[N][2][-1,:,:], rho_analytical, data_dict[N][1][-1,:], data_dict[N][0][-1])}')
    print(f'delta_C = {delta_by_C_norm(data_dict[N][2][-1,:,:], rho_analytical) / delta_by_C_norm(0, rho_analytical)}')
    print(f'delta_L = {delta_by_L1_norm(data_dict[N][2][-1,:,:], rho_analytical, data_dict[N][1][-1,:], data_dict[N][0][-1])/ delta_by_L1_norm(np.zeros_like(rho_analytical), rho_analytical, data_dict[N][1][-1,:], data_dict[N][0][-1])}')

N = 100
Delta_C = 0.010393873116349983
Delta_L = 8.78883115295408e-05
delta_C = 0.005033532667842327
delta_L = 4.7356660260064706e-05
N = 200
Delta_C = 0.010353873116349721
Delta_L = 4.8248763541692576e-05
delta_C = 0.005014161514811963
delta_L = 2.595836745723418e-05
N = 400
Delta_C = 0.010353873116349721
Delta_L = 2.9918915601118978e-05
delta_C = 0.005014161514811963
delta_L = 1.6084562368247704e-05
N = 800
Delta_C = 0.0103438731163501
Delta_L = 2.1669822701508595e-05
delta_C = 0.005009318726554586
delta_L = 1.1645417095410632e-05
N = 1600
Delta_C = 0.0103438731163501
Delta_L = 1.760838006505338e-05
delta_C = 0.005009318726554586
delta_L = 9.461006706564005e-06
